In [9]:
import os
os.getcwd()

'c:\\Users\\avant\\OneDrive\\Desktop\\VSCode\\C++\\SpikeMutations\\notebooks'

In [6]:
import pandas as pd

# Load embeddings
embeddings_file = "../data/processed/embeddings.csv"
embeddings = pd.read_csv(embeddings_file)

print("Shape of embeddings:", embeddings.shape)
print("\nPreview:")
print(embeddings.head(3))

# Check for missing values
nan_counts = embeddings.isna().sum().sum()
print(f"\nTotal missing values in embeddings: {nan_counts}")

# Check if all rows have expected dimensionality
expected_dim = embeddings.shape[1] - 1  # minus mut_id column
bad_rows = embeddings.drop("mut_id", axis=1).isna().any(axis=1).sum()
print(f"Rows with missing embedding values: {bad_rows}")

# Check uniqueness of mut_id
duplicates = embeddings["mut_id"].duplicated().sum()
print(f"Duplicate mutation IDs: {duplicates}")

# Summary
if nan_counts == 0 and duplicates == 0 and bad_rows == 0:
    print("\nEmbeddings file looks good! All mutations have proper vectors.")
else:
    print("\nSomething is off, check the counts above carefully.")


Shape of embeddings: (4221, 1281)

Preview:
  mut_id         0         1         2         3         4         5  \
0  N331A -0.067741  0.199019 -0.179698  0.201105 -0.067334  0.090759   
1  N331C -0.064530  0.072357 -0.115226  0.166408 -0.056582  0.087773   
2  N331D -0.019751  0.020047 -0.132962  0.280015 -0.045891  0.089194   

          6         7         8  ...      1270      1271      1272      1273  \
0  0.039440 -0.037416 -0.094640  ...  0.088068  0.020139  0.111115  0.164220   
1  0.107421  0.028012 -0.098189  ... -0.144662  0.109343  0.224311  0.116104   
2  0.085321  0.080868 -0.093845  ...  0.040745  0.110557  0.165327  0.120866   

       1274      1275      1276      1277      1278      1279  
0 -0.012009  0.274097  0.053875 -0.067687 -0.159784  0.017576  
1  0.093836  0.178063  0.139208 -0.079412 -0.243861  0.061967  
2  0.029265  0.243903  0.033439 -0.036450  0.051138  0.127455  

[3 rows x 1281 columns]

Total missing values in embeddings: 257280
Rows with missing emb

In [8]:
import pandas as pd
import numpy as np
import os

emb_file = "../data/processed/embeddings.csv"
labels_file = "../data/processed/labels.csv"

emb = pd.read_csv(emb_file, index_col=False)

# ensure mut_id column exists (if it was saved as index the first col may be unnamed)
if "mut_id" not in emb.columns:
    first = emb.columns[0]
    emb = emb.rename(columns={first: "mut_id"})

# detect embedding columns (everything except mut_id)
embedding_cols = [c for c in emb.columns if c != "mut_id"]

# rows where all embedding cols are NaN
rows_allnan = emb[embedding_cols].isna().all(axis=1)
num_rows_allnan = rows_allnan.sum()
total_nans = emb[embedding_cols].isna().sum().sum()

print("Shape of embeddings:", emb.shape)
print("Rows with ALL-NaN embeddings:", num_rows_allnan)
print("Total NaN values in embedding matrix:", total_nans)
print("Sanity check (rows * dims) == total_nans? ->",
      f"{num_rows_allnan} * {len(embedding_cols)} = {num_rows_allnan * len(embedding_cols)}")

nan_mut_ids = emb.loc[rows_allnan, "mut_id"].tolist()
print("\nFirst 30 mut_ids with all-NaN embeddings:", nan_mut_ids[:30])

# Check if they contain stop-codon symbol '*' (very common cause)
num_with_star = sum(1 for m in nan_mut_ids if "*" in str(m))
print("Of those, how many contain '*' (stop-codon):", num_with_star)

# Save the list for inspection
os.makedirs("data/processed", exist_ok=True)
pd.Series(nan_mut_ids, name="mut_id").to_csv("../data/processed/embeddings_allnan_mutids.csv", index=False)
print("Saved list to data/processed/embeddings_allnan_mutids.csv")


Shape of embeddings: (4221, 1281)
Rows with ALL-NaN embeddings: 201
Total NaN values in embedding matrix: 257280
Sanity check (rows * dims) == total_nans? -> 201 * 1280 = 257280

First 30 mut_ids with all-NaN embeddings: ['N331*', 'I332*', 'T333*', 'N334*', 'L335*', 'C336*', 'P337*', 'F338*', 'G339*', 'E340*', 'V341*', 'F342*', 'N343*', 'A344*', 'T345*', 'R346*', 'F347*', 'A348*', 'S349*', 'V350*', 'Y351*', 'A352*', 'W353*', 'N354*', 'R355*', 'K356*', 'R357*', 'I358*', 'S359*', 'N360*']
Of those, how many contain '*' (stop-codon): 201
Saved list to data/processed/embeddings_allnan_mutids.csv


In [10]:
import pandas as pd

embeddings_file = "../data/processed/embeddings.csv"
emb = pd.read_csv(embeddings_file)

# Identify embedding columns (exclude mut_id)
embedding_cols = [c for c in emb.columns if c != "mut_id"]

# Compute min and max across all embeddings (excluding NaNs)
min_val = emb[embedding_cols].min().min()
max_val = emb[embedding_cols].max().max()

print(f"Minimum embedding value: {min_val}")
print(f"Maximum embedding value: {max_val}")

# Optional: count how many exact zeros there are
num_zeros = (emb[embedding_cols] == 0.0).sum().sum()
print(f"Number of embedding values that are exactly 0.0: {num_zeros}")


Minimum embedding value: -8.983174324035645
Maximum embedding value: 1.6193392276763916
Number of embedding values that are exactly 0.0: 0


#### Script 004_prepare_dataset.py

In [2]:
import pandas as pd

# File paths
files = {
    "labels": "../data/processed/labels.csv",
    "seq_features": "../data/processed/seq_features.csv",
    "embeddings": "../data/processed/embeddings.csv",
    "mutation_features": "../data/processed/mutation_features.csv",
    "final_dataset_clean": "../data/processed/final_dataset_clean.csv"
}

# Load all files
dfs = {}
for name, path in files.items():
    try:
        dfs[name] = pd.read_csv(path)
    except Exception as e:
        dfs[name] = str(e)

dfs.keys(), {k: (v.shape if isinstance(v, pd.DataFrame) else v) for k,v in dfs.items()}


(dict_keys(['labels', 'seq_features', 'embeddings', 'mutation_features', 'final_dataset_clean']),
 {'labels': (4221, 14),
  'seq_features': (4221, 11),
  'embeddings': (4221, 1281),
  'mutation_features': (4221, 1301),
  'final_dataset_clean': (4221, 2590)})

In [7]:
import pandas as pd
files = {
    "labels": "../data/processed/labels.csv",
    "seq_features": "../data/processed/seq_features.csv",
    "embeddings": "../data/processed/embeddings.csv",
}
datacols = {name: pd.read_csv(path).columns.tolist() for name, path in files.items()}
for name, cols in datacols.items():
    print(f"{name} columns ({len(cols)}): {cols}\n")

labels columns (14): ['site_RBD', 'site_SARS2', 'wildtype', 'mutant', 'mutation', 'mutation_RBD', 'bind_lib1', 'bind_lib2', 'ace2_score', 'expr_lib1', 'expr_lib2', 'expr_score', 'mut_id', 'label']

seq_features columns (11): ['mut_id', 'pos', 'wt', 'mut', 'blosum62', 'grantham', 'delta_charge', 'delta_hydro', 'delta_volume', 'delta_polarity', 'is_stop']

embeddings columns (1281): ['mut_id', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '

In [13]:
import pandas as pd

# --- Load all datasets ---
labels = pd.read_csv("../data/processed/labels.csv")
embeddings = pd.read_csv("../data/processed/embeddings.csv")
seq_features = pd.read_csv("../data/processed/seq_features.csv")

# --- Step 1: Identify silent mutations (WT == Mutant) ---
silent_mask = labels["wildtype"] == labels["mutant"]
silent_mutations = labels[silent_mask].copy()
print(f"Total silent mutations: {silent_mutations.shape[0]}")

# --- Step 2: Merge silent mutations with embeddings & seq_features ---
silent_full = (
    silent_mutations
    .merge(embeddings, on="mut_id", how="left")
    .merge(seq_features, on="mut_id", how="left", suffixes=("", "_seq"))
)

# --- Step 3: Check NaN counts ---
nan_counts = silent_full.isna().sum()

# --- Step 4: Check for all-zero embeddings ---
embedding_cols = [str(i) for i in range(1280)]  # 0 ... 1279
is_all_zero = (silent_full[embedding_cols] == 0.0).all(axis=1)
silent_full["all_zero_embedding"] = is_all_zero

# --- Step 5: Summaries ---
print("\n=== Silent mutation NaN summary ===")
print(nan_counts[nan_counts > 0])

print(f"\nSilent mutations with all-zero embeddings: {is_all_zero.sum()}")

# --- Step 6: Save details for inspection ---
silent_full.to_csv("Same_wildtype_mutants.csv", index=False)
print("Saved detailed report to Same_wildtype_mutants.csv")


Total silent mutations: 201

=== Silent mutation NaN summary ===
Series([], dtype: int64)

Silent mutations with all-zero embeddings: 0
Saved detailed report to Same_wildtype_mutants.csv


In [14]:
import pandas as pd

# --- Load all datasets ---
labels = pd.read_csv("../data/processed/labels.csv")
embeddings = pd.read_csv("../data/processed/embeddings.csv")
seq_features = pd.read_csv("../data/processed/seq_features.csv")

# --- Step 1: Identify silent mutations (WT == Mutant) ---
silent_mask = labels["mutant"] == "*"
silent_mutations = labels[silent_mask].copy()
print(f"Total silent mutations: {silent_mutations.shape[0]}")

# --- Step 2: Merge silent mutations with embeddings & seq_features ---
silent_full = (
    silent_mutations
    .merge(embeddings, on="mut_id", how="left")
    .merge(seq_features, on="mut_id", how="left", suffixes=("", "_seq"))
)

# --- Step 3: Check NaN counts ---
nan_counts = silent_full.isna().sum()

# --- Step 4: Check for all-zero embeddings ---
embedding_cols = [str(i) for i in range(1280)]  # 0 ... 1279
is_all_zero = (silent_full[embedding_cols] == 0.0).all(axis=1)
silent_full["all_zero_embedding"] = is_all_zero

# --- Step 5: Summaries ---
print("\n=== Silent mutation NaN summary ===")
print(nan_counts[nan_counts > 0])

print(f"\nSilent mutations with all-zero embeddings: {is_all_zero.sum()}")

# --- Step 6: Save details for inspection ---
silent_full.to_csv("StopCodons.csv", index=False)
print("Saved detailed report to StopCodons.csv")


Total silent mutations: 201

=== Silent mutation NaN summary ===
bind_lib1     201
bind_lib2     201
ace2_score    201
expr_lib1       3
expr_lib2       2
             ... 
1275          201
1276          201
1277          201
1278          201
1279          201
Length: 1286, dtype: int64

Silent mutations with all-zero embeddings: 0
Saved detailed report to StopCodons.csv


In [15]:
import pandas as pd

# --- Load datasets ---
labels = pd.read_csv("../data/processed/labels.csv")
embeddings = pd.read_csv("../data/processed/embeddings.csv")
seq_features = pd.read_csv("../data/processed/seq_features.csv")

# --- Step 1: Identify silent mutations (WT == Mutant) ---
silent_mask = labels["wildtype"] == labels["mutant"]
silent_mutations = labels[silent_mask].copy()
silent_mutations["mutation_type"] = "silent"

# --- Step 2: Identify stop codons (mutant == '*') ---
stop_mask = labels["mutant"] == "*"
stop_mutations = labels[stop_mask].copy()
stop_mutations["mutation_type"] = "stop"

# --- Step 3: Combine ---
special_mutations = pd.concat([silent_mutations, stop_mutations], ignore_index=True)

print(f"Silent mutations: {silent_mutations.shape[0]}")
print(f"Stop codons: {stop_mutations.shape[0]}")
print(f"Total special cases: {special_mutations.shape[0]}")

# --- Step 4: Merge with embeddings and seq_features ---
special_full = (
    special_mutations
    .merge(embeddings, on="mut_id", how="left")
    .merge(seq_features, on="mut_id", how="left", suffixes=("", "_seq"))
)

# --- Step 5: Check NaN counts ---
nan_counts = special_full.isna().sum()

# --- Step 6: Check for all-zero embeddings ---
embedding_cols = [str(i) for i in range(1280)]  # 0..1279
is_all_zero = (special_full[embedding_cols] == 0.0).all(axis=1)
special_full["all_zero_embedding"] = is_all_zero

# --- Step 7: Needs repair flag ---
# If all-zero embeddings OR NaNs present
special_full["needs_repair"] = is_all_zero | special_full.isna().any(axis=1)

# --- Step 8: Summaries ---
print("\n=== Special mutation NaN summary ===")
print(nan_counts[nan_counts > 0])

print(f"\nSilent mutations with all-zero embeddings: {special_full.query('mutation_type == \"silent\"')['all_zero_embedding'].sum()}")
print(f"Stop codons with all-zero embeddings: {special_full.query('mutation_type == \"stop\"')['all_zero_embedding'].sum()}")

print(f"\nMutations flagged as needs_repair: {special_full['needs_repair'].sum()}")

# --- Step 9: Save detailed report ---
special_full.to_csv("special_mutations_check.csv", index=False)
print("Saved detailed report to special_mutations_check.csv")


Silent mutations: 201
Stop codons: 201
Total special cases: 402

=== Special mutation NaN summary ===
bind_lib1     201
bind_lib2     201
ace2_score    201
expr_lib1       3
expr_lib2       2
             ... 
1275          201
1276          201
1277          201
1278          201
1279          201
Length: 1286, dtype: int64

Silent mutations with all-zero embeddings: 0
Stop codons with all-zero embeddings: 0

Mutations flagged as needs_repair: 201
Saved detailed report to special_mutations_check.csv
